In [0]:
# Notebook autonome : il porte ses propres %run et se lance seul.
# Relancés par main_translations, ils sont sans effet de bord (idempotents).

In [0]:
%run ./env

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./translation_function

# dim_trad_language

> **Nom** : préfixée `dim_trad_` comme les autres tables de cette phase, et pour
> ne pas entrer en collision avec la table `dim_language` déjà présente dans le
> schéma `common`, qui relève d'un autre sujet et ne doit pas être touchée.

Référentiel des langues du modèle de traduction, alimenté depuis `parameters_languages`.

Cette table porte le filtre RLS : c'est la **seule** table sur laquelle un rôle de
sécurité est écrit. Elle propage ensuite son filtre à toutes les tables de
traduction via les relations 1 -> * sur `language`.

## Rapprochement avec USERCULTURE()

`USERCULTURE()` renvoie une culture complète (`fr-FR`, `cs-CZ`, ...) dont la
partie langue suit la norme **ISO 639-1**. La colonne `code` de
`parameters_languages` ne la suit pas partout : elle porte le code **pays** pour
l'ukrainien (`UA` au lieu de `uk`) et le tchèque (`CZ` au lieu de `cs`).

Un rapprochement direct sur `code` basculerait donc silencieusement ces deux
langues sur le repli anglais. On construit une colonne dédiée `culture_code`
via un mapping explicite, jamais dérivée de `code`.

> Toute nouvelle langue ajoutée par le front doit être ajoutée à `CULTURE_MAP`.
> À défaut elle tombera sur le repli anglais (dégradé, mais pas cassant).

In [0]:
# Périmètre du projet : 4 langues.
# Ce dictionnaire fait deux choses à la fois :
#   - il DÉCLARE les langues supportées (celles absentes ici sont hors périmètre) ;
#   - il fait correspondre le code métier au code langue ISO 639-1 renvoyé par
#     USERCULTURE(), qui n'est pas le même pour le tchèque.
# Ajouter une langue au projet = ajouter une ligne ici, rien d'autre.
CULTURE_MAP = {
    "FR": "fr",
    "EN": "en",
    "RO": "ro",
    "CZ": "cs",   # tchèque : ISO 639-1 = cs, la source porte le code pays CZ
}

# Hors périmètre pour l'instant : PL (polonais) et UA (ukrainien), présents dans
# parameters_languages mais non traités par le projet. Attention si on les ajoute :
# l'ukrainien est "uk" en ISO 639-1, pas "ua".

# Langue de repli quand la culture de l'utilisateur n'est pas dans le périmètre
FALLBACK_LANGUAGE_CODE = "en"

culture_expr = F.create_map([F.lit(x) for x in sum(CULTURE_MAP.items(), ())])

In [0]:
dim_trad_language = (
    spark.table(f"{source_catalog}.parameters_languages")
    .filter(F.col("deleted") == False)
    .withColumn("culture_code", culture_expr[F.col("code")])
    # Restreint au périmètre du projet : une langue absente de CULTURE_MAP a un
    # culture_code null et n'entre pas dans le modèle. C'est ce filtre qui pilote
    # tout le reste : le produit cartésien de build_translations porte sur
    # dim_trad_language, donc les tables de traduction ne contiendront que ces langues.
    .filter(F.col("culture_code").isNotNull())
    .select(
        F.col("id_parameter_language").alias("language"),
        F.col("code"),
        F.col("name").alias("language_label"),
        "culture_code",
    )
)

## Garde-fou : détecter une langue non mappée

Si une langue est ajoutée dans `parameters_languages` sans être déclarée dans
`CULTURE_MAP`, `culture_code` sera `null` et cette langue deviendra
inatteignable par le RLS. On le rend visible au rafraîchissement plutôt que de
le découvrir en production.

In [0]:
# Langues présentes en base mais hors périmètre du projet : informatif, pas une alerte.
langues_hors_perimetre = (
    spark.table(f"{source_catalog}.parameters_languages")
    .filter(F.col("deleted") == False)
    .filter(~F.col("code").isin(list(CULTURE_MAP.keys())))
    .select("id_parameter_language", "code", "name")
)

if verbose_mode == 'debug':
    print(f"Périmètre du projet : {len(CULTURE_MAP)} langues -> {list(CULTURE_MAP.keys())}")
    display(dim_trad_language)

    if langues_hors_perimetre.count() > 0:
        print("Langues présentes en base mais hors périmètre (non traduites) :")
        display(langues_hors_perimetre)

In [0]:
# La langue de repli doit exister, sinon le garde-fou du rôle RLS ne protège rien
if dim_trad_language.filter(F.col("culture_code") == FALLBACK_LANGUAGE_CODE).count() == 0:
    raise ValueError(
        f"La langue de repli '{FALLBACK_LANGUAGE_CODE}' est absente de dim_trad_language : "
        "le rôle RLS renverrait un rapport vide pour toute culture non reconnue."
    )

Import dim_trad_language

In [0]:
current_process = "dim_trad_language"

In [0]:
# Cible : le catalogue commun, pas le schéma du projet.
target_dim_trad_language = f"{translations_schema}.{current_process}"
print(target_dim_trad_language)

In [0]:
all_columns = dim_trad_language.columns
display(all_columns)

In [0]:
# define the primary key
primary_key = ['language']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    print(additional_columns)

In [0]:
ensure_delta_table(dim_trad_language, target_dim_trad_language)

# mode="full" imposé, et non execution_mode : dim_trad_language est une table de
# référence entièrement redérivée à chaque run. En mode "update", une langue
# retirée de parameters_languages resterait indéfiniment dans le modèle et
# continuerait d'être proposée au RLS.
handle_table_update(
    dim_trad_language,
    target_dim_trad_language,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
    )